# Football Scouting & Player Recommendation System
## Exploratory Data Analysis & Machine Learning Walkthrough

This notebook demonstrates the end-to-end data science process for the Football Scouting and Player Recommendation System. This workflow includes:
1. **Data Loading & Inspection**: Examining the structure and characteristics of the raw dataset.
2. **Data Imputation & Preprocessing**: Handling missing values using position-wise medians and standardizing features.
3. **Exploratory Data Analysis (EDA)**: Visualizing demographic distributions, correlation matrices, and metrics of interest.
4. **K-Means Clustering**: Assigning players to tactical archetypes based on performance stats.
5. **Player Recommendation**: Building a cosine similarity engine to identify scouts and targets.

In [ ]:
# Append parent directory to path to allow importing src modules
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

sns.set_theme(style="whitegrid")
%matplotlib inline

### 1. Load and Inspect Raw Data

In [ ]:
raw_path = '../data/players.csv'
df_raw = pd.read_csv(raw_path)

print(f"Raw dataset shape: {df_raw.shape}")
df_raw.info()

In [ ]:
print("Missing values per column:")
print(df_raw.isnull().sum())

print("\nSample player records with missing values:")
df_raw[df_raw.isnull().any(axis=1)].head()

### 2. Run Preprocessing Pipeline
We impute missing values. We use **position-wise median imputation** because outfield positions have vastly different stats. Imputing a defender's missing tackles with the global median (which is dragged down by forwards) would skew their profile.

In [ ]:
from src.preprocessing import run_preprocessing_pipeline

# Run the full pipeline
df_clean, scaled_df = run_preprocessing_pipeline(
    raw_path='../data/players.csv',
    clean_path='../data/players_clean.csv',
    scaler_path='../models/scaler.pkl'
)

print(f"\nCleaned data missing values verification: {df_clean.isnull().sum().sum()}")
df_clean.head()

### 3. Exploratory Data Analysis (EDA)
Let's look at the distribution of positions, age, market values, and attribute correlations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Position distribution
sns.countplot(data=df_clean, x='position', ax=axes[0], palette='viridis')
axes[0].set_title('Player Count by Position')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Count')

# Age distribution
sns.histplot(data=df_clean, x='age', hue='position', multiple='stack', kde=True, ax=axes[1], palette='tab10')
axes[1].set_title('Age Distribution by Position')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Top Performers by Metrics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top Goal Scorers
top_goals = df_clean.sort_values(by='goals', ascending=False).head(10)
sns.barplot(data=top_goals, x='goals', y='player_name', hue='position', ax=axes[0], dodge=False, palette='coolwarm')
axes[0].set_title('Top 10 Goal Scorers')
axes[0].set_xlabel('Goals')
axes[0].set_ylabel('Player Name')

# Top Playmakers
top_assists = df_clean.sort_values(by='assists', ascending=False).head(10)
sns.barplot(data=top_assists, x='assists', y='player_name', hue='position', ax=axes[1], dodge=False, palette='mako')
axes[1].set_title('Top 10 Assist Providers')
axes[1].set_xlabel('Assists')
axes[1].set_ylabel('Player Name')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
plt.figure(figsize=(10, 8))
from src.preprocessing import ML_FEATURES
corr = df_clean[ML_FEATURES + ['age', 'market_value']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Player Feature Correlation Matrix')
plt.show()

### 4. Player Clustering & Archetypes (K-Means)
We use K-Means with K=5 to segment players based on their stats, then assign tactical labels based on the average traits of each cluster.

In [ ]:
from src.clustering import perform_clustering, get_cluster_stats

# Run clustering
df_clustered, kmeans_model, archetypes_mapping = perform_clustering(df_clean, scaled_df, n_clusters=5)

print("Archetype Mapping:")
for cid, label in archetypes_mapping.items():
    print(f"Cluster {cid} -> {label}")

In [ ]:
# Cluster statistics breakdown
stats = get_cluster_stats(df_clustered)
stats

In [ ]:
# Principal Component Analysis (PCA) for cluster visualization
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca_res = pca.fit_transform(scaled_df)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=pca_res[:, 0], 
    y=pca_res[:, 1], 
    hue=df_clustered['cluster_label'], 
    palette='Set1', 
    alpha=0.8, 
    edgecolor='k'
)
plt.title('Player Clusters Visualized in 2D space (PCA)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance Explained)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance Explained)')
plt.legend(title='Tactical Archetype')
plt.show()

### 5. Similarity Engine & Player Recommendations
We use cosine similarity on the scaled performance vectors to find similar profiles.

In [ ]:
from src.recommendation import PlayerRecommender

# Initialize recommender
recommender = PlayerRecommender(df_clustered, scaled_df)

# Select a random target player to recommendation
target_name = df_clustered.iloc[10]['player_name']
print(f"Finding 5 players similar to: {target_name} ({df_clustered.iloc[10]['position']}, Value: €{df_clustered.iloc[10]['market_value']}M)\n")

recs = recommender.get_recommendations(target_name, top_n=5)
for r in recs:
    print(f"- {r['recommended_player']} | Sim Score: {r['similarity_score']*100:.1f}% | Club: {r['details']['club']} | Position: {r['details']['position']} | Value: €{r['details']['market_value']}M")

In [ ]:
# Attribute explanation breakdown
top_match_name = recs[0]['recommended_player']
print(f"Stat Comparison explaining similarity between {target_name} & {top_match_name}:\n")

expl = recommender.explain_similarity(target_name, top_match_name)
expl_df = pd.DataFrame(expl).T
expl_df